# ARIA v3.0: The Living Auditor - Dynamic Risk Monitoring System

**Captain's Log**: Building a real-time disaster monitoring system that integrates Week 3-4 shelter risk data with dynamic rainfall monitoring during Typhoon Fung-wong.

**Mission**: Create an interactive dashboard that answers "Where is the most dangerous place RIGHT NOW?"

In [59]:
# Import required libraries
import os
import json
import requests
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import HeatMap
from dotenv import load_dotenv
import numpy as np
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("✅ Libraries loaded successfully")
print(f"📊 APP_MODE: {os.getenv('APP_MODE')}")
print(f"🎯 TARGET_COUNTY: {os.getenv('TARGET_COUNTY')}")

✅ Libraries loaded successfully
📊 APP_MODE: SIMULATION
🎯 TARGET_COUNTY: 花蓮縣


In [60]:
def load_week3_shelter_data():
    """
    Load Week 3 shelter risk audit data.
    """
    print("Loading Week 3 shelter risk data...")
    
    try:
        with open('shelter_risk_audit.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Convert to DataFrame
        df = pd.DataFrame(data)
        
        # Create geometry from TWD97 coordinates (EPSG:3826)
        geometry = [Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])]
        
        # Create GeoDataFrame with EPSG:3826 (TWD97)
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:3826')
        
        # Standardize risk level names
        risk_mapping = {'safe': 'LOW', 'high': 'HIGH'}
        gdf['river_risk'] = gdf['risk_level'].map(risk_mapping).fillna('LOW')
        
        print(f"Loaded {len(gdf)} shelters from Week 3 data")
        print(f"CRS: {gdf.crs}")
        print(f"River Risk Distribution: {gdf['river_risk'].value_counts().to_dict()}")
        
        return gdf
        
    except FileNotFoundError:
        print("Week 3 shelter data not found - using sample data")
        return create_sample_shelter_data()
    except Exception as e:
        print(f"Error loading Week 3 data: {e}")
        return create_sample_shelter_data()

def load_week4_terrain_data():
    """
    Load Week 4 terrain risk audit data.
    """
    print("Loading Week 4 terrain risk data...")
    
    try:
        with open('terrain_risk_audit.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Process terrain data
        processed_data = []
        for item in data:
            terrain_stats = item.get('terrain_stats', {})
            processed_data.append({
                'shelter_id': item['shelter_id'],
                'name': item['name'],
                'longitude': item['longitude'],
                'latitude': item['latitude'],
                'mean_elevation': terrain_stats.get('mean_elevation_m', 0),
                'max_slope': terrain_stats.get('max_slope_deg', 0),
                'river_distance_m': item.get('river_distance_m', 0),
                'terrain_risk_raw': item.get('risk_level', 'LOW')
            })
        
        # Convert to DataFrame
        df = pd.DataFrame(processed_data)
        
        # Create geometry from TWD97 coordinates (EPSG:3826)
        geometry = [Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])]
        
        # Create GeoDataFrame with EPSG:3826 (TWD97)
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:3826')
        
        # Standardize terrain risk names
        risk_mapping = {'高風險': 'HIGH', '極高風險': 'HIGH', '低風險': 'LOW'}
        gdf['terrain_risk'] = gdf['terrain_risk_raw'].map(risk_mapping).fillna('LOW')
        
        print(f"Loaded {len(gdf)} terrain assessments from Week 4 data")
        print(f"CRS: {gdf.crs}")
        print(f"Terrain Risk Distribution: {gdf['terrain_risk'].value_counts().to_dict()}")
        
        return gdf
        
    except FileNotFoundError:
        print("Week 4 terrain data not found - using sample data")
        return create_sample_shelter_data()
    except Exception as e:
        print(f"Error loading Week 4 data: {e}")
        return create_sample_shelter_data()

def merge_week3_week4_data(week3_gdf, week4_gdf):
    """
    Merge Week 3 and Week 4 data by shelter name.
    """
    print("Merging Week 3-4 data...")
    
    # Use spatial join to match shelters by location (more reliable than name matching)
    merged_gdf = gpd.sjoin(week3_gdf, week4_gdf, how='left', predicate='intersects')
    
    # Clean up columns
    merged_gdf = merged_gdf.drop(columns=['index_right'], errors='ignore')
    
    # Fill missing terrain risk with default values
    merged_gdf['terrain_risk'] = merged_gdf['terrain_risk'].fillna('LOW')
    merged_gdf['mean_elevation'] = merged_gdf['mean_elevation'].fillna(0)
    merged_gdf['max_slope'] = merged_gdf['max_slope'].fillna(0)
    
    # Select and rename key columns
    final_columns = [
        'name', 'county', 'capacity', 'river_risk', 'terrain_risk',
        'mean_elevation', 'max_slope', 'river_distance_m', 'geometry'
    ]
    
    # Ensure all required columns exist
    for col in final_columns:
        if col not in merged_gdf.columns:
            merged_gdf[col] = None
    
    shelters_gdf = merged_gdf[final_columns].copy()
    
    print(f"Merged data: {len(shelters_gdf)} shelters")
    print(f"Combined Risk Distribution:")
    print(f"   River Risk: {shelters_gdf['river_risk'].value_counts().to_dict()}")
    print(f"   Terrain Risk: {shelters_gdf['terrain_risk'].value_counts().to_dict()}")
    
    return shelters_gdf

def create_sample_shelter_data():
    """
    Create sample shelter data as fallback.
    """
    print("Creating fallback sample data...")
    
    sample_data = [
        {
            'name': '花蓮市避難所1',
            'county': '花蓮縣',
            'river_risk': 'HIGH',
            'terrain_risk': 'HIGH',
            'mean_elevation': 150.0,
            'max_slope': 20.0,
            'river_distance_m': 0.0,
            'longitude': 311766.9938203232,
            'latitude': 2653051.9139427906
        }
    ]
    
    df = pd.DataFrame(sample_data)
    geometry = [Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])]
    return gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:3826')

def create_shelter_gdf():
    """
    Create integrated shelter GeoDataFrame from Week 3-4 data.
    """
    # Load Week 3 and Week 4 data
    week3_gdf = load_week3_shelter_data()
    week4_gdf = load_week4_terrain_data()
    
    # Merge datasets
    shelters_gdf = merge_week3_week4_data(week3_gdf, week4_gdf)
    
    # Filter for target counties (Hualien and Yilan focus)
    target_counties = ['花蓮縣', '宜蘭縣']
    if 'county' in shelters_gdf.columns:
        filtered_gdf = shelters_gdf[shelters_gdf['county'].str.contains('|'.join(target_counties), na=False)]
    else:
        filtered_gdf = shelters_gdf
    
    print(f"Filtered to {len(filtered_gdf)} shelters in target area")
    
    return filtered_gdf

In [61]:
# Load shelter data from Week 3-4 integration
shelters_gdf = create_shelter_gdf()

print(f"\\nShelter Risk Distribution:")
print(f"   River Risk - HIGH: {(shelters_gdf['river_risk'] == 'HIGH').sum()}")
print(f"   River Risk - MEDIUM: {(shelters_gdf['river_risk'] == 'MEDIUM').sum()}")
print(f"   River Risk - LOW: {(shelters_gdf['river_risk'] == 'LOW').sum()}")
print(f"   Terrain Risk - HIGH: {(shelters_gdf['terrain_risk'] == 'HIGH').sum()}")
print(f"   Terrain Risk - MEDIUM: {(shelters_gdf['terrain_risk'] == 'MEDIUM').sum()}")
print(f"   Terrain Risk - LOW: {(shelters_gdf['terrain_risk'] == 'LOW').sum()}")

# Display sample of loaded data
if not shelters_gdf.empty:
    print(f"\\nSample shelter data:")
    display(shelters_gdf[['name', 'county', 'river_risk', 'terrain_risk', 'mean_elevation', 'max_slope']].head())

Loading Week 3 shelter risk data...
Loaded 5888 shelters from Week 3 data
CRS: EPSG:3826
River Risk Distribution: {'LOW': 3320, 'HIGH': 2568}
Loading Week 4 terrain risk data...
Loaded 10 terrain assessments from Week 4 data
CRS: EPSG:3826
Terrain Risk Distribution: {'HIGH': 10}
Merging Week 3-4 data...
Merged data: 5888 shelters
Combined Risk Distribution:
   River Risk: {'LOW': 3320, 'HIGH': 2568}
   Terrain Risk: {'LOW': 5888}
Filtered to 446 shelters in target area
\nShelter Risk Distribution:
   River Risk - HIGH: 210
   River Risk - MEDIUM: 0
   River Risk - LOW: 236
   Terrain Risk - HIGH: 0
   Terrain Risk - MEDIUM: 0
   Terrain Risk - LOW: 446
\nSample shelter data:


,name,county,river_risk,terrain_risk,mean_elevation,max_slope
31,None,花蓮縣秀林鄉,LOW,LOW,0.0,0.0
1287,None,花蓮縣富里鄉,HIGH,LOW,0.0,0.0
1313,None,花蓮縣富里鄉,HIGH,LOW,0.0,0.0
1323,None,花蓮縣富里鄉,HIGH,LOW,0.0,0.0
1331,None,花蓮縣富里鄉,HIGH,LOW,0.0,0.0


In [62]:
def normalize_cwa_json(data):
    """
    Normalize CWA JSON data from both LIVE API and CoLife formats.
    
    Returns: List of dictionaries with standardized format
    """
    normalized = []
    
    try:
        stations = data['records']['Station']
    except KeyError:
        print("Invalid JSON format - missing records.Station")
        return normalized
    
    for station in stations:
        try:
            # Handle coordinate differences between CWA API and CoLife
            if 'GeoInfo' in station and 'Coordinates' in station['GeoInfo']:
                coords = station['GeoInfo']['Coordinates']
                if isinstance(coords, list) and len(coords) >= 2:
                    # CWA API format - take WGS84 coordinates (index 1)
                    lon, lat = coords[1]
                else:
                    # CoLife format - single coordinate pair (WGS84 only)
                    lon, lat = coords
            else:
                continue
            
            # Extract rainfall data - handle both string and numeric formats
            rain_1hr = station.get('Rainfall1hr', 0)
            if isinstance(rain_1hr, str):
                rain_1hr = float(rain_1hr) if rain_1hr.replace('.', '').isdigit() else 0
            
            # Filter out invalid data
            if rain_1hr == -998 or rain_1hr < 0:
                continue
            
            normalized.append({
                'station_name': station.get('StationName', 'Unknown'),
                'lat': float(lat),
                'lon': float(lon),
                'rain_1hr': rain_1hr,
                'county': station.get('County', '')
            })
            
        except (KeyError, ValueError, TypeError) as e:
            continue
    
    return normalized

def fetch_rainfall_data():
    """
    Fetch rainfall data based on APP_MODE setting.
    """
    app_mode = os.getenv('APP_MODE', 'SIMULATION')
    
    if app_mode == 'LIVE':
        print("LIVE Mode: Fetching from CWA API...")
        try:
            api_key = os.getenv('CWA_API_KEY')
            if not api_key or api_key == 'your-key-here':
                raise ValueError("CWA_API_KEY not configured")
            
            url = f"https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001?Authorization={api_key}"
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            
            data = response.json()
            normalized = normalize_cwa_json(data)
            print(f"Fetched {len(normalized)} stations from CWA API")
            return normalized
            
        except Exception as e:
            print(f"API call failed: {e}")
            print("Falling back to simulation mode...")
            app_mode = 'SIMULATION'
    
    if app_mode == 'SIMULATION':
        print("SIMULATION Mode: Loading Typhoon Fung-wong data...")
        try:
            sim_file = os.getenv('SIMULATION_DATA', 'fungwong_202511.json')
            with open(sim_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            normalized = normalize_cwa_json(data)
            print(f"Loaded {len(normalized)} stations from simulation data")
            return normalized
            
        except FileNotFoundError:
            print("Simulation file not found - creating sample data")
            return create_sample_data()
        except Exception as e:
            print(f"Error loading simulation data: {e}")
            return create_sample_data()
    
    return []

def create_sample_data():
    """
    Create sample rainfall data for testing when no data is available.
    """
    print("Creating sample rainfall data...")
    sample_stations = [
        {'station_name': 'Su-ao', 'lat': 24.5935, 'lon': 121.8195, 'rain_1hr': 130.5, 'county': '宜蘭縣'},
        {'station_name': 'Nan-ao', 'lat': 24.4667, 'lon': 121.7833, 'rain_1hr': 95.2, 'county': '宜蘭縣'},
        {'station_name': 'Hualien', 'lat': 23.9817, 'lon': 121.6044, 'rain_1hr': 65.8, 'county': '花蓮縣'},
        {'station_name': 'Yuli', 'lat': 23.3364, 'lon': 121.3117, 'rain_1hr': 45.3, 'county': '花蓮縣'},
        {'station_name': 'Taitung', 'lat': 22.7518, 'lon': 121.1539, 'rain_1hr': 25.7, 'county': '台東縣'},
    ]
    return sample_stations

def create_rainfall_gdf(rainfall_data):
    """
    Create GeoDataFrame from rainfall station data.
    """
    if not rainfall_data:
        print("No rainfall data available")
        return gpd.GeoDataFrame()
    
    # Create DataFrame
    df = pd.DataFrame(rainfall_data)
    
    # Create geometry column
    geometry = [Point(lon, lat) for lon, lat in zip(df['lon'], df['lat'])]
    
    # Create GeoDataFrame with EPSG:4326 (WGS84)
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
    
    # Convert to EPSG:3826 for spatial analysis
    gdf_3826 = gdf.to_crs('EPSG:3826')
    
    print(f"Created rainfall GeoDataFrame: {len(gdf)} stations")
    print(f"CRS: {gdf.crs} → {gdf_3826.crs}")
    
    return gdf_3826

# Execute rainfall data loading
print("=" * 50)
print("LOADING RAINFALL DATA")
print("=" * 50)

# Test data fetching
rainfall_data = fetch_rainfall_data()
print(f"\\nTotal stations processed: {len(rainfall_data)}")
if rainfall_data:
    print(f"Sample station: {rainfall_data[0]['station_name']} - {rainfall_data[0]['rain_1hr']}mm/hr")

# Create rainfall GeoDataFrame
rainfall_gdf = create_rainfall_gdf(rainfall_data)

print(f"\\nRainfall data ready: {len(rainfall_gdf)} stations")
print("=" * 50)

LOADING RAINFALL DATA
SIMULATION Mode: Loading Typhoon Fung-wong data...
Simulation file not found - creating sample data
Creating sample rainfall data...
\nTotal stations processed: 5
Sample station: Su-ao - 130.5mm/hr
Created rainfall GeoDataFrame: 5 stations
CRS: EPSG:4326 → EPSG:3826
\nRainfall data ready: 5 stations


In [63]:
def assess_dynamic_risk(shelters_gdf, rainfall_gdf, affected_shelters_gdf):
    """
    Apply dynamic risk classification logic.
    """
    # Create a copy of all shelters to add risk assessment
    shelters_with_risk = shelters_gdf.copy()
    
    # Initialize dynamic risk as SAFE for all shelters
    shelters_with_risk['dynamic_risk'] = 'SAFE'
    shelters_with_risk['nearest_rainfall'] = 0.0
    shelters_with_risk['nearest_station'] = 'None'
    
    if affected_shelters_gdf.empty:
        print("No affected shelters - all shelters marked as SAFE")
        return shelters_with_risk
    
    # Assess risk for affected shelters
    for idx, shelter in affected_shelters_gdf.iterrows():
        rain_1hr = shelter['rain_1hr']
        terrain_risk = shelter['terrain_risk']
        station_name = shelter['station_name']
        
        # Dynamic risk logic
        if rain_1hr > 80:
            dynamic_risk = 'CRITICAL'
        elif rain_1hr > 40 and terrain_risk == 'HIGH':
            dynamic_risk = 'URGENT'
        elif rain_1hr > 40 or terrain_risk == 'HIGH':
            dynamic_risk = 'WARNING'
        else:
            dynamic_risk = 'SAFE'
        
        # Update the shelter's risk assessment
        shelter_idx = shelters_with_risk[shelters_with_risk['name'] == shelter['name']].index
        if len(shelter_idx) > 0:
            shelters_with_risk.loc[shelter_idx[0], 'dynamic_risk'] = dynamic_risk
            shelters_with_risk.loc[shelter_idx[0], 'nearest_rainfall'] = rain_1hr
            shelters_with_risk.loc[shelter_idx[0], 'nearest_station'] = station_name
    
    # Print risk distribution
    print("Dynamic Risk Assessment:")
    risk_counts = shelters_with_risk['dynamic_risk'].value_counts()
    for risk_level, count in risk_counts.items():
        print(f"   {risk_level}: {count} shelters")
    
    return shelters_with_risk

In [66]:
def create_rainfall_impact_zones(rainfall_gdf, buffer_km=5):
    """
    Create buffer zones around high rainfall stations.
    """
    if rainfall_gdf.empty:
        print("❌ No rainfall data for impact analysis")
        return gpd.GeoDataFrame()
    
    # Filter for significant rainfall (> 20mm/hr)
    significant_rain = rainfall_gdf[rainfall_gdf['rain_1hr'] > 20].copy()
    
    if significant_rain.empty:
        print("⚠️ No stations with significant rainfall (> 20mm/hr)")
        return gpd.GeoDataFrame()
    
    # Create 5km buffer (5000 meters in EPSG:3826)
    significant_rain['geometry'] = significant_rain['geometry'].buffer(buffer_km * 1000)
    
    print(f"🌧️ Created impact zones for {len(significant_rain)} stations (>20mm/hr)")
    print(f"📏 Buffer radius: {buffer_km} km")
    
    return significant_rain

def find_affected_shelters(shelters_gdf, impact_zones_gdf):
    """
    Find shelters within rainfall impact zones using spatial join.
    """
    if shelters_gdf.empty or impact_zones_gdf.empty:
        print("❌ Cannot perform spatial join - empty data")
        return gpd.GeoDataFrame()
    
    # CRS check - critical for accurate spatial join
    if str(shelters_gdf.crs) != str(impact_zones_gdf.crs):
        print(f"❌ CRS MISMATCH! Shelters: {shelters_gdf.crs}, Zones: {impact_zones_gdf.crs}")
        return gpd.GeoDataFrame()
    
    # Perform spatial join
    affected_shelters = gpd.sjoin(shelters_gdf, impact_zones_gdf, how='inner', predicate='within')
    
    if affected_shelters.empty:
        print("⚠️ No shelters found within rainfall impact zones")
        return gpd.GeoDataFrame()
    
    # Clean up column names
    affected_shelters = affected_shelters.drop(columns=['index_right'], errors='ignore')
    
    print(f"🎯 Found {len(affected_shelters)} shelters within impact zones")
    
    return affected_shelters

# Create impact zones and find affected shelters
impact_zones_gdf = create_rainfall_impact_zones(rainfall_gdf)
affected_shelters_gdf = find_affected_shelters(shelters_gdf, impact_zones_gdf)

if not affected_shelters_gdf.empty:
    print(f"\n🏫 Affected Shelters Summary:")
    for idx, shelter in affected_shelters_gdf.iterrows():
        print(f"   {shelter['name']} - Rain: {shelter['rain_1hr']:.1f}mm/hr")

🌧️ Created impact zones for 5 stations (>20mm/hr)
📏 Buffer radius: 5 km
🎯 Found 99 shelters within impact zones

🏫 Affected Shelters Summary:
   None - Rain: 45.3mm/hr
   None - Rain: 45.3mm/hr
   None - Rain: 45.3mm/hr
   None - Rain: 45.3mm/hr
   None - Rain: 45.3mm/hr
   None - Rain: 45.3mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr
   None - Rain: 65.8mm/hr


In [67]:
# Perform dynamic risk assessment
shelters_with_risk_gdf = assess_dynamic_risk(shelters_gdf, rainfall_gdf, affected_shelters_gdf)

# Show high-risk shelters
high_risk = shelters_with_risk_gdf[shelters_with_risk_gdf['dynamic_risk'].isin(['CRITICAL', 'URGENT'])]
if not high_risk.empty:
    print(f"\\nHIGH RISK SHELTERS:")
    for idx, shelter in high_risk.iterrows():
        print(f"   {shelter['name']} - {shelter['dynamic_risk']} ({shelter['nearest_rainfall']:.1f}mm/hr)")
else:
    print(f"\\nNo critical or urgent risk shelters identified")

Dynamic Risk Assessment:
   SAFE: 446 shelters
\nNo critical or urgent risk shelters identified


def create_rainfall_impact_zones(rainfall_gdf, buffer_km=5):
    """
    Create buffer zones around high rainfall stations.
    """
    if rainfall_gdf.empty:
        print("No rainfall data for impact analysis")
        return gpd.GeoDataFrame()
    
    # Filter for significant rainfall (> 20mm/hr)
    significant_rain = rainfall_gdf[rainfall_gdf['rain_1hr'] > 20].copy()
    
    if significant_rain.empty:
        print("No stations with significant rainfall (> 20mm/hr)")
        return gpd.GeoDataFrame()
    
    # Create 5km buffer (5000 meters in EPSG:3826)
    significant_rain['geometry'] = significant_rain['geometry'].buffer(buffer_km * 1000)
    
    print(f"Created impact zones for {len(significant_rain)} stations (>20mm/hr)")
    print(f"Buffer radius: {buffer_km} km")
    
    return significant_rain

def find_affected_shelters(shelters_gdf, impact_zones_gdf):
    """
    Find shelters within rainfall impact zones using spatial join.
    """
    if shelters_gdf.empty or impact_zones_gdf.empty:
        print("Cannot perform spatial join - empty data")
        return gpd.GeoDataFrame()
    
    # CRS check - critical for accurate spatial join
    if str(shelters_gdf.crs) != str(impact_zones_gdf.crs):
        print(f"CRS MISMATCH! Shelters: {shelters_gdf.crs}, Zones: {impact_zones_gdf.crs}")
        return gpd.GeoDataFrame()
    
    # Perform spatial join
    affected_shelters = gpd.sjoin(shelters_gdf, impact_zones_gdf, how='inner', predicate='within')
    
    if affected_shelters.empty:
        print("No shelters found within rainfall impact zones")
        return gpd.GeoDataFrame()
    
    # Clean up column names
    affected_shelters = affected_shelters.drop(columns=['index_right'], errors='ignore')
    
    print(f"Found {len(affected_shelters)} shelters within impact zones")
    
    return affected_shelters

# Create impact zones and find affected shelters
impact_zones_gdf = create_rainfall_impact_zones(rainfall_gdf)
affected_shelters_gdf = find_affected_shelters(shelters_gdf, impact_zones_gdf)

if not affected_shelters_gdf.empty:
    print(f"\\nAffected Shelters Summary:")
    for idx, shelter in affected_shelters_gdf.iterrows():
        print(f"   {shelter['name']} - Rain: {shelter['rain_1hr']:.1f}mm/hr")

## 5. Dynamic Risk Assessment

**Captain's Log**: Applying the dynamic risk classification logic that combines rainfall intensity with pre-existing terrain risks.

In [70]:
def assess_dynamic_risk(shelters_gdf, rainfall_gdf, affected_shelters_gdf):
    """
    Apply dynamic risk classification logic.
    """
    # Create a copy of all shelters to add risk assessment
    shelters_with_risk = shelters_gdf.copy()
    
    # Initialize dynamic risk as SAFE for all shelters
    shelters_with_risk['dynamic_risk'] = 'SAFE'
    shelters_with_risk['nearest_rainfall'] = 0.0
    shelters_with_risk['nearest_station'] = 'None'
    
    if affected_shelters_gdf.empty:
        print("⚠️ No affected shelters - all shelters marked as SAFE")
        return shelters_with_risk
    
    # Assess risk for affected shelters
    for idx, shelter in affected_shelters_gdf.iterrows():
        rain_1hr = shelter['rain_1hr']
        terrain_risk = shelter['terrain_risk']
        station_name = shelter['station_name']
        
        # Dynamic risk logic
        if rain_1hr > 80:
            dynamic_risk = 'CRITICAL'
        elif rain_1hr > 40 and terrain_risk == 'HIGH':
            dynamic_risk = 'URGENT'
        elif rain_1hr > 40 or terrain_risk == 'HIGH':
            dynamic_risk = 'WARNING'
        else:
            dynamic_risk = 'SAFE'
        
        # Update the shelter's risk assessment
        shelter_idx = shelters_with_risk[shelters_with_risk['name'] == shelter['name']].index
        if len(shelter_idx) > 0:
            shelters_with_risk.loc[shelter_idx[0], 'dynamic_risk'] = dynamic_risk
            shelters_with_risk.loc[shelter_idx[0], 'nearest_rainfall'] = rain_1hr
            shelters_with_risk.loc[shelter_idx[0], 'nearest_station'] = station_name
    
    # Print risk distribution
    print(f"🚨 Dynamic Risk Assessment:")
    risk_counts = shelters_with_risk['dynamic_risk'].value_counts()
    for risk_level, count in risk_counts.items():
        print(f"   {risk_level}: {count} shelters")
    
    return shelters_with_risk

# Perform dynamic risk assessment
shelters_with_risk_gdf = assess_dynamic_risk(shelters_gdf, rainfall_gdf, affected_shelters_gdf)

# Show high-risk shelters
high_risk = shelters_with_risk_gdf[shelters_with_risk_gdf['dynamic_risk'].isin(['CRITICAL', 'URGENT'])]
if not high_risk.empty:
    print(f"\n🚨 HIGH RISK SHELTERS:")
    for idx, shelter in high_risk.iterrows():
        print(f"   {shelter['name']} - {shelter['dynamic_risk']} ({shelter['nearest_rainfall']:.1f}mm/hr)")

🚨 Dynamic Risk Assessment:
   SAFE: 446 shelters


# Create the interactive map
print("Creating interactive map...")
aria_map = create_interactive_map(rainfall_gdf, shelters_with_risk_gdf)
print("Interactive map created successfully!")

In [71]:
def create_interactive_map(rainfall_gdf, shelters_with_risk_gdf):
    """
    Create interactive Folium map with rainfall and shelter layers.
    """
    # Calculate center point (focus on target area)
    if not shelters_with_risk_gdf.empty:
        # Convert to EPSG:4326 for Folium
        shelters_4326 = shelters_with_risk_gdf.to_crs('EPSG:4326')
        center_lat = shelters_4326.geometry.y.mean()
        center_lon = shelters_4326.geometry.x.mean()
    else:
        # Default to Hualien area
        center_lat, center_lon = 23.9817, 121.6044
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=9,
        tiles='OpenStreetMap'
    )
    
    # Add rainfall stations layer
    if not rainfall_gdf.empty:
        rainfall_4326 = rainfall_gdf.to_crs('EPSG:4326')
        
        for idx, station in rainfall_4326.iterrows():
            rain_1hr = station['rain_1hr']
            
            # Determine color based on rainfall intensity
            if rain_1hr > 80:
                color = 'red'
                radius = 15
            elif rain_1hr > 40:
                color = 'orange'
                radius = 12
            elif rain_1hr > 20:
                color = 'yellow'
                radius = 10
            else:
                color = 'green'
                radius = 8
            
            folium.CircleMarker(
                location=[station.geometry.y, station.geometry.x],
                radius=radius,
                popup=f"<b>{station['station_name']}</b><br>"
                      f"時雨量: {rain_1hr:.1f} mm/hr<br>"
                      f"縣市: {station['county']}",
                color='black',
                fill=True,
                fillColor=color,
                fillOpacity=0.7,
                weight=2
            ).add_to(m)
    
    # Add shelters layer
    if not shelters_with_risk_gdf.empty:
        shelters_4326 = shelters_with_risk_gdf.to_crs('EPSG:4326')
        
        for idx, shelter in shelters_4326.iterrows():
            # Determine color based on dynamic risk
            risk_colors = {
                'CRITICAL': 'darkred',
                'URGENT': 'red',
                'WARNING': 'orange',
                'SAFE': 'green'
            }
            
            color = risk_colors.get(shelter['dynamic_risk'], 'blue')
            
            # Create detailed popup
            popup_content = f"""
            <b>Shelter: {shelter['name']}</b><br>
            <hr>
            <b>Dynamic Risk Level:</b> <span style="color:{color}; font-weight:bold;">{shelter['dynamic_risk']}</span><br>
            <b>Terrain Risk:</b> {shelter['terrain_risk']} (slope: {shelter['max_slope']:.1f}°)<br>
            <b>River Risk:</b> {shelter['river_risk']}<br>
            <b>Elevation:</b> {shelter['mean_elevation']:.1f} m<br>
            <b>River Distance:</b> {shelter['river_distance_m']:.0f} m<br>
            <b>Nearest Station:</b> {shelter['nearest_station']}<br>
            <b>Hourly Rain:</b> {shelter['nearest_rainfall']:.1f} mm/hr
            """
            
            folium.Marker(
                location=[shelter.geometry.y, shelter.geometry.x],
                popup=folium.Popup(popup_content, max_width=300),
                icon=folium.Icon(
                    color=color,
                    icon='home' if shelter['dynamic_risk'] == 'SAFE' else 'exclamation-triangle',
                    prefix='fa'
                )
            ).add_to(m)
    
    # Add heatmap layer
    if not rainfall_gdf.empty:
        rainfall_4326 = rainfall_gdf.to_crs('EPSG:4326')
        heat_data = [
            [station.geometry.y, station.geometry.x, station['rain_1hr']]
            for idx, station in rainfall_4326.iterrows()
            if station['rain_1hr'] > 0
        ]
        
        if heat_data:
            HeatMap(
                heat_data,
                name='Rainfall Heatmap',
                radius=15,
                blur=10,
                gradient={
                    0.2: 'blue',
                    0.4: 'cyan',
                    0.6: 'yellow',
                    0.8: 'orange',
                    1.0: 'red'
                }
            ).add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Add title
    title_html = '''
    <h3 align="center" style="font-size:16px"><b>ARIA v3.0 - Dynamic Risk Monitoring System</b></h3>
    <h4 align="center" style="font-size:12px">Typhoon Fung-wong Real-time Assessment (Week 3-4 Integration)</h4>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    print(f"Interactive map created with center at [{center_lat:.4f}, {center_lon:.4f}]")
    
    return m

# Create the interactive map
aria_map = create_interactive_map(rainfall_gdf, shelters_with_risk_gdf)

Interactive map created with center at [24.2746, 121.6412]


# Save the interactive map
output_file = 'ARIA_v3_Fungwong.html'
aria_map.save(output_file)
print(f"Interactive map saved as: {output_file}")

# Display the map in notebook
print("\\nARIA v3.0 Dynamic Risk Monitoring System")
print("=" * 50)
print(f"Total Rainfall Stations: {len(rainfall_gdf)}")
print(f"Total Shelters: {len(shelters_with_risk_gdf)}")
print(f"Affected Shelters: {len(affected_shelters_gdf)}")

if not shelters_with_risk_gdf.empty:
    risk_summary = shelters_with_risk_gdf['dynamic_risk'].value_counts()
    print(f"\\nRisk Distribution:")
    for risk, count in risk_summary.items():
        print(f"   {risk}: {count} shelters")

print(f"\\nARIA v3.0 system ready! Open {output_file} in your browser.")
print(f"System Features:")
print(f"   Week 3-4 shelter risk data integration")
print(f"   Real-time rainfall monitoring (LIVE/SIMULATION modes)")
print(f"   Dynamic risk assessment (CRITICAL/URGENT/WARNING/SAFE)")
print(f"   Interactive map with heatmaps and detailed popups")

# Display the map
display(aria_map)

In [72]:
# Save the interactive map
output_file = 'ARIA_v3_Fungwong.html'
aria_map.save(output_file)
print(f"💾 Interactive map saved as: {output_file}")

# Display the map in notebook
print("\n🗺️ ARIA v3.0 Dynamic Risk Monitoring System")
print("=" * 50)
print(f"📊 Total Rainfall Stations: {len(rainfall_gdf)}")
print(f"🏫 Total Shelters: {len(shelters_with_risk_gdf)}")
print(f"🎯 Affected Shelters: {len(affected_shelters_gdf)}")

if not shelters_with_risk_gdf.empty:
    risk_summary = shelters_with_risk_gdf['dynamic_risk'].value_counts()
    print(f"\n🚨 Risk Distribution:")
    for risk, count in risk_summary.items():
        print(f"   {risk}: {count} shelters")

print(f"\n✅ ARIA v3.0 system ready! Open {output_file} in your browser.")

# Display the map
display(aria_map)

💾 Interactive map saved as: ARIA_v3_Fungwong.html

🗺️ ARIA v3.0 Dynamic Risk Monitoring System
📊 Total Rainfall Stations: 5
🏫 Total Shelters: 446
🎯 Affected Shelters: 99

🚨 Risk Distribution:
   SAFE: 446 shelters

✅ ARIA v3.0 system ready! Open ARIA_v3_Fungwong.html in your browser.


## 8. Bonus: AI Tactical Advisor (Optional)

**Captain's Log**: Integrating Gemini AI for tactical disaster response recommendations.

In [ ]:
# AI Tactical Advisor Bonus Feature

# First, let's create some test critical shelters for demonstration
print("🧪 Creating test critical shelters for AI advisor demo...")
test_indices = shelters_with_risk_gdf.head(3).index
shelters_with_risk_gdf.loc[test_indices[0], 'dynamic_risk'] = 'CRITICAL'
shelters_with_risk_gdf.loc[test_indices[0], 'nearest_rainfall'] = 130.5
shelters_with_risk_gdf.loc[test_indices[0], 'nearest_station'] = 'Su-ao'

shelters_with_risk_gdf.loc[test_indices[1], 'dynamic_risk'] = 'URGENT'
shelters_with_risk_gdf.loc[test_indices[1], 'nearest_rainfall'] = 95.2
shelters_with_risk_gdf.loc[test_indices[1], 'nearest_station'] = 'Nan-ao'

shelters_with_risk_gdf.loc[test_indices[2], 'dynamic_risk'] = 'CRITICAL'
shelters_with_risk_gdf.loc[test_indices[2], 'nearest_rainfall'] = 65.8
shelters_with_risk_gdf.loc[test_indices[2], 'nearest_station'] = 'Hualien'

print("✅ Test critical shelters created!")
print(f"   {shelters_with_risk_gdf.loc[test_indices[0], 'name']} - CRITICAL (130.5mm/hr)")
print(f"   {shelters_with_risk_gdf.loc[test_indices[1], 'name']} - URGENT (95.2mm/hr)")
print(f"   {shelters_with_risk_gdf.loc[test_indices[2], 'name']} - CRITICAL (65.8mm/hr)")

# Now run the AI advisor
try:
    import google.generativeai as genai
    
    # Configure Gemini API (you need to set GEMINI_API_KEY in .env)
    gemini_api_key = os.getenv('GEMINI_API_KEY')
    if gemini_api_key and gemini_api_key != 'your-gemini-key-here':
        genai.configure(api_key=gemini_api_key)
        model = genai.GenerativeModel('gemini-pro')
        
        # Get top 3 most critical shelters
        critical_shelters = shelters_with_risk_gdf[
            shelters_with_risk_gdf['dynamic_risk'].isin(['CRITICAL', 'URGENT'])
        ].head(3)
        
        if not critical_shelters.empty:
            print("\n🤖 AI Tactical Advisor Analysis:")
            print("=" * 40)
            
            for idx, shelter in critical_shelters.iterrows():
                prompt = f"""
                你是花蓮縣防災指揮中心的專家顧問。以下是鳳凰颱風期間的即時數據：
                避難所: {shelter['name']}
                地形風險: {shelter['terrain_risk']} (最大坡度: {shelter['max_slope']:.1f}°)
                河川風險: {shelter['river_risk']}
                海拔高度: {shelter['mean_elevation']:.1f}m
                最近雨量站: {shelter['nearest_station']} (時雨量: {shelter['nearest_rainfall']:.1f}mm)
                動態風險等級: {shelter['dynamic_risk']}
                
                請以3句話給出指揮官的緊急應變建議。
                """
                
                response = model.generate_content(prompt)
                print(f"\n🏫 {shelter['name']} ({shelter['dynamic_risk']}):")
                print(response.text)
        
        else:
            print("\n✅ No critical shelters requiring AI advisor consultation.")
    
    else:
        print("\n⚠️ GEMINI_API_KEY not configured - skipping AI advisor feature.")
        print("💡 To use AI advisor:")
        print("   1. Get a Gemini API key from: https://makersuite.google.com/app/apikey")
        print("   2. Add to .env file: GEMINI_API_KEY=your-actual-key-here")
        print("   3. Re-run this cell")
        
except ImportError:
    print("\n⚠️ google-generativeai not installed - run: pip install google-generativeai")
except Exception as e:
    print(f"\n❌ AI Advisor error: {e}")